In [3]:
import pandas as pd

# Input files
full_file = "Full.csv"
clean_file = "Full_cleaned.csv"

# Output file
output_file = "Full_updated.csv"

# Read files
df_full = pd.read_csv(full_file)
df_clean = pd.read_csv(clean_file)

# Create a mapping: title -> Reason Remove
reason_map = (
    df_clean.dropna(subset=["title"])
    .drop_duplicates(subset=["title"], keep="first")
    .set_index("title")["Reason Remove"]
)

# Update Reason Remove in Full based on matching title
df_full["Reason Remove"] = df_full["title"].map(reason_map).combine_first(
    df_full["Reason Remove"]
)

# Save result
df_full.to_csv(output_file, index=False)

print(f"Done! Saved to: {output_file}")
print(f"Full rows: {len(df_full)}")
print(f"Matched titles: {df_full['title'].isin(reason_map.index).sum()}")


Done! Saved to: Full_updated.csv
Full rows: 1352
Matched titles: 696


In [4]:
df_clean = pd.read_csv('Full_updated.csv')
count_full = (df_full["Reason Remove"].astype(str).str.strip() == "o0o0o").sum()

In [5]:
count_full

np.int64(335)

In [7]:
import pandas as pd
import re

# Input / output
input_file = "Full_updated.csv"
output_file = "Full_updated_clinical.csv"

# Read CSV
df = pd.read_csv(input_file)

# More specific clinical-related keywords
clinical_keywords = [
    # Clinical setting / healthcare
    "clinical",
    "clinically",
    "patient",
    "patients",
    "hospital",
    "hospitals",
    "healthcare",
    "health care",
    "medical",
    "medicine",
    "physician",
    "clinician",
    "clinical practice",

    # Electronic health records
    "EHR",
    "EHRs",
    "EMR",
    "EMRs",
    "electronic health record",
    "electronic health records",
    "electronic medical record",
    "electronic medical records",
    "medical record",
    "medical records",
    "health record",
    "health records",

    # Clinical studies / trials
    "clinical trial",
    "clinical trials",
    "clinical study",
    "clinical studies",
    "randomized controlled trial",
    "randomised controlled trial",
    "randomized clinical trial",
    "randomised clinical trial",
    "RCT",
    "RCTs",

    # Clinical research design
    "cohort",
    "cohort study",
    "cohort studies",
    "case report",
    "case reports",
    "case series",
    "prospective study",
    "prospective studies",
    "retrospective study",
    "retrospective studies",
    "longitudinal study",
    "longitudinal studies",

    # Diagnosis / screening
    "diagnosis",
    "diagnostic",
    "diagnostics",
    "screening",
    "clinical diagnosis",
    "clinical diagnostic",

    # Treatment / intervention
    "treatment",
    "treatments",
    "therapeutic",
    "therapeutics",
    "therapy",
    "therapies",
    "intervention",
    "interventions",

    # Clinical outcomes
    "clinical outcome",
    "clinical outcomes",
    "patient outcome",
    "patient outcomes",
    "treatment outcome",
    "treatment outcomes",

    # Clinical safety / adverse events
    "adverse event",
    "adverse events",
    "side effect",
    "side effects",

    # Clinical population / care
    "inpatient",
    "inpatients",
    "outpatient",
    "outpatients",
    "hospitalization",
    "hospitalisation",
    "hospitalized",
    "hospitalised",
]

# Remove duplicate keywords and sort by length
clinical_keywords = sorted(
    set(clinical_keywords),
    key=len,
    reverse=True
)

# Build regex
pattern = r"\b(?:%s)\b" % "|".join(
    re.escape(keyword) for keyword in clinical_keywords
)

# Condition 1:
# title contains a clinical-related keyword
title_match = (
    df["title"]
    .fillna("")
    .astype(str)
    .str.contains(
        pattern,
        case=False,
        regex=True
    )
)

# Condition 2:
# Reason Remove is NOT "o0o0o"
reason_not_o0o0o = (
    df["Reason Remove"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    != "o0o0o"
)

# Final mask
mask = title_match & reason_not_o0o0o

# Change Reason Remove to clinical
df.loc[mask, "Reason Remove"] = "Clinical"

# Save
df.to_csv(output_file, index=False)

print(f"Done! Saved to: {output_file}")
print(f"Rows changed to clinical: {mask.sum()}")

Done! Saved to: Full_updated_clinical.csv
Rows changed to clinical: 340


In [17]:
import pandas as pd

# Input file
input_file = "Full_updated_clinical.csv"

# Output file
output_file = "Full_removed.csv"

# Read CSV
df = pd.read_csv(input_file)

# Keep rows where Reason Remove is NOT "o0o0o"
filtered_df = df[
    df["Reason Remove"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower() != "o0o0o"
]

# Save
filtered_df.to_csv(output_file, index=False)

print(f"Done!")
print(f"Original rows: {len(df)}")
print(f"Filtered rows: {len(filtered_df)}")
print(f"Removed rows: {len(df) - len(filtered_df)}")
print(f"Saved to: {output_file}")

Done!
Original rows: 1352
Filtered rows: 1019
Removed rows: 333
Saved to: Full_removed.csv


In [16]:
import pandas as pd

# Input file
input_file = "Full_updated_clinical.csv"

# Output file
output_file = "Full_o0o0o.csv"

# Read CSV
df = pd.read_csv(input_file)

# Keep only rows where Reason Remove == "o0o0o"
filtered_df = df[
    df["Reason Remove"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower() == "o0o0o"
]

# Save
filtered_df.to_csv(output_file, index=False)

print("Done!")
print(f"Total rows: {len(filtered_df)}")
print(f"Saved to: {output_file}")

Done!
Total rows: 333
Saved to: Full_o0o0o.csv


In [15]:

import pandas as pd
from rapidfuzz.fuzz import ratio
from itertools import combinations

# Input file
input_file = "Full_updated_clinical.csv"

# Read CSV
df = pd.read_csv(input_file)

# Normalize title
df["title_normalized"] = (
    df["title"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

results = []

# Compare every pair of titles
for i, j in combinations(range(len(df)), 2):

    title1 = df.loc[i, "title_normalized"]
    title2 = df.loc[j, "title_normalized"]

    # Skip empty titles
    if not title1 or not title2:
        continue

    similarity = ratio(title1, title2)

    # Keep only similarity >= 80%
    if similarity >= 80:
        results.append({
            "similarity": round(similarity, 2),
            "row_1": i,
            "row_2": j,
            "title_1": df.loc[i, "title"],
            "title_2": df.loc[j, "title"]
        })


# Convert to DataFrame
results_df = pd.DataFrame(results)

# Sort highest similarity first
if not results_df.empty:
    results_df = results_df.sort_values(
        by="similarity",
        ascending=False
    )

    print(f"Found {len(results_df)} similar title pairs:\n")

    for _, row in results_df.iterrows():
        print("=" * 100)
        print(f"Similarity: {row['similarity']}%")
        print(f"Row {row['row_1']}: {row['title_1']}")
        print(f"Row {row['row_2']}: {row['title_2']}")
else:
    print("No title pairs with similarity >= 80% found.")


# Save results
results_df.to_csv(
    "duplicate_titles_80_percent.csv",
    index=False
)

print("\nSaved to: duplicate_titles_80_percent.csv")

Found 23 similar title pairs:

Similarity: 100.0%
Row 80: A ReAct Agentic AI System for Natural Language Querying and Statistical Analysis of The Cancer Genome Atlas Clinical Data
Row 142: A ReAct Agentic AI System for Natural Language Querying and Statistical Analysis of The Cancer Genome Atlas Clinical Data
Similarity: 98.59%
Row 658: Large Language Model Agent for Modular Task Execution in Drug Discovery.
Row 831: Large Language Model Agent for Modular TaskExecution in Drug Discovery
Similarity: 96.68%
Row 185: AI mirrors experimental science to uncover a mechanism of gene transfer crucial to bacterial evolution.
Row 186: AI mirrors experimental science to uncover a novel mechanism of gene transfer crucial to bacterial evolution
Similarity: 95.76%
Row 692: Not Just One Agent: LLM-Based Multi-Agent Systems for Medicine from Answer Generation to Accountable Workflow Orchestration
Row 693: Not Just One Agent: Multi-Agent Systems for Medicine from Answer Generation to Accountable Workfl

In [20]:

import pandas as pd
from rapidfuzz.fuzz import ratio
from itertools import combinations

# Input file
input_file = "Full_o0o0o.csv"

# Read CSV
df = pd.read_csv(input_file)

# Normalize title
df["title_normalized"] = (
    df["title"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

results = []

# Compare every pair of titles
for i, j in combinations(range(len(df)), 2):

    title1 = df.loc[i, "title_normalized"]
    title2 = df.loc[j, "title_normalized"]

    # Skip empty titles
    if not title1 or not title2:
        continue

    similarity = ratio(title1, title2)

    # Keep only similarity >= 80%
    if similarity >= 70:
        results.append({
            "similarity": round(similarity, 2),
            "row_1": i,
            "row_2": j,
            "title_1": df.loc[i, "title"],
            "title_2": df.loc[j, "title"]
        })


# Convert to DataFrame
results_df = pd.DataFrame(results)

# Sort highest similarity first
if not results_df.empty:
    results_df = results_df.sort_values(
        by="similarity",
        ascending=False
    )

    print(f"Found {len(results_df)} similar title pairs:\n")

    for _, row in results_df.iterrows():
        print("=" * 100)
        print(f"Similarity: {row['similarity']}%")
        print(f"Row {row['row_1']}: {row['title_1']}")
        print(f"Row {row['row_2']}: {row['title_2']}")
else:
    print("No title pairs with similarity >= 80% found.")


# Save results
results_df.to_csv(
    "duplicate_titles_70_percent.csv",
    index=False
)

print("\nSaved to: duplicate_titles_70_percent.csv")

Found 5 similar title pairs:

Similarity: 77.99%
Row 299: An agentic framework for autonomous scientific discovery in cancer pathology.
Row 303: DERMA-Agent: An Agentic Framework for Prognostic Discovery in Pan-Cancer Pathology
Similarity: 77.13%
Row 154: AutoProteinEngine: A Large Language Model Driven Agent Framework for Multimodal AutoML in Protein Engineering
Row 161: TourSynbio-Search: A Large Language Model Driven Agent Framework for Unified Search Method for Protein Engineering
Similarity: 76.82%
Row 152: The Virtual Lab of AI agents designs new SARS-CoV-2 nanobodies.
Row 160: The Virtual Lab: AI Agents Design New SARS-CoV-2 Nanobodies with Experimental Validation
Similarity: 75.41%
Row 142: Accelerating Scientific Discovery with Autonomous Goal-evolving Agents
Row 185: Accelerating scientific discovery with Co-Scientist.
Similarity: 70.18%
Row 68: SpaCellAgent: A Self-Evolving LLM-Based Multi-Agent Framework for Trajectory Analysis
Row 117: CellAgent: An LLM-driven Multi-Agent 

In [23]:
df = pd.read_csv('/home/phpham/Work/BioAgents/codedebug/title_primary_category.csv')

In [24]:
stats = df["primary_category"].value_counts()

print(stats)

primary_category
drug_discovery                        106
transcriptomics                        60
cross_domain_bioinformatics            57
proteomics                             41
computational_pathology_bioimaging     36
genomics                               32
epigenomics                             2
Name: count, dtype: int64
